### 1. Import libraries and Reproducibility rules

In [4]:
import json
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt


In [5]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")
print("Pytorch version:", torch.__version__)

Pytorch version: 2.8.0+cpu


### 2. Preprocessing

In [8]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=Mean, std=STD)
])

train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(50, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.RandomResizedCrop(
        200, scale=(0.9, 1.0), ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

### 3. Data loading

In [11]:
train_dir = "data/train"
evaluation_dir = "data/test"

evaluation_dataset = datasets.ImageFolder(evaluation_dir, transform=evaluation_transform)
evaluation_loader = DataLoader(evaluation_dataset, batch_size=20, shuffle=False, num_workers=0)

loader_generator = torch.Generator().manual_seed(SEED)

def make_train_loader(transform):
    dataset = datasets.ImageFolder(train_dir, transform=transform)
    return DataLoader(dataset, batch_size=20, shuffle=True, num_workers=0, generator=loader_generator)

baseline_train_loader = make_train_loader(evaluation_transform)

print("Classes:", evaluation_dataset.class_to_idx)
print("Training images:", len(baseline_train_loader.dataset))
print("Evaluation images:", len(evaluation_dataset))

Classes: {'curly': 0, 'straight': 1}
Training images: 800
Evaluation images: 201


### 4. Modelling

The model will be made with said pipeline :
1. Conv layer : Scans the image for patterns 
2. ReLU + MaxPool : Shrinks image and keeps strong signals
3. Flatten: Lines the numbers up in one long row
4. Hidden Layer: Combines the patterns
5. Output Layer: the numbers 

In [13]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

model = Model().to(device)
print(model)

Model(
  (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (hidden): Linear(in_features=313632, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=1, bias=True)
)


### Q1 

The answer is ``nn.BCEwithLogitsLoss()``. it applies to sigmoid function then measures the error